<a href="https://colab.research.google.com/github/Vedant864/RAG-pipeline-from-scratch/blob/main/RAG_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installing Libraries

!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q pymupdf
!pip install -q torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135

In [2]:
import pymupdf
import chromadb
import torch
import numpy as np
import pandas as pd
import textwrap
from sentence_transformers import SentenceTransformer

from transformers import (
AutoTokenizer,

AutoModelForCausalLM,

pipeline
)

In [3]:
print("="*80)

print("LOADING EMBEDDING MODEL")

print("="*80)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(" Embedding Model Loaded Successfully!")

LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding Model Loaded Successfully!


In [4]:
#LOAD MISTRAL MODEL

print("\n")
print("="*80)
print("LOADING MISTRAL MODEL")
print("="*80)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16)

print(" Mistral Model Loaded Successfully!")



LOADING MISTRAL MODEL


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

 Mistral Model Loaded Successfully!


In [6]:
#Create Text Generation Pipeline

print("\n")
print("CREATING GENERATION PIPELINE")

generator = pipeline(
   "text-generation",
model=model,
tokenizer=tokenizer,
max_new_tokens=256,
temperature=0.3,
pad_token_id=tokenizer.eos_token_id
)
print(" Pipeline Ready!")

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.




CREATING GENERATION PIPELINE
 Pipeline Ready!


In [9]:
# INITIALIZE CHROMADB

print("\n")
print("="*80)
print("INITIALIZING CHROMADB")
print("="*80)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_rag_collection", get_or_create=True)

print("ChromaDB Collection Created!")



INITIALIZING CHROMADB
ChromaDB Collection Created!


In [15]:
# Loading Documents
import os
import urllib.request

print("\n")
print("="*80)
print("LOADING PDF DOCUMENT")
print("="*80)

PDF_PATH="/content/Attention paper.pdf"

# Automatically download the PDF if it does not exist
if not os.path.exists(PDF_PATH):
    print("PDF file not found locally. Downloading 'Attention Is All You Need' paper...")
    url = "https://arxiv.org/pdf/1706.03762.pdf"
    urllib.request.urlretrieve(url, PDF_PATH)
    print("Download completed!")

document = pymupdf.open(PDF_PATH)
print(" PDF Loaded Successfully!")
print(" Total Pages:", len(document))



LOADING PDF DOCUMENT
PDF file not found locally. Downloading 'Attention Is All You Need' paper...
Download completed!
 PDF Loaded Successfully!
 Total Pages: 15


In [24]:
# EXTRACT TEXT FROM PDF

print("\n")
print("="*80)
print("EXTRACTING TEXT")
print("="*80)

full_text = ""

for page_number in range(len(document)):
    page = document[page_number]
    # Extract the text from the page and assign it to the variable 'text'
    text = page.get_text()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    full_text += text + " "

print(" Text Extraction Completed!")
print(" Total Characters:", len(full_text))



EXTRACTING TEXT
 Text Extraction Completed!
 Total Characters: 39496


In [25]:
#PREVIEW DOCUMENT TEXT

print("\n")

print("="*80)

print("DOCUMENT PREVIEW")

print("="*80)

print(full_text[:1500])



DOCUMENT PREVIEW
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convoluti

In [39]:
#TEXT CHUNKING

print("\n")
print("="*80)
print("CREATING TEXT CHUNKS")
print("="*80)

def chunk_text(text, chunk_size=700, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks



CREATING TEXT CHUNKS
